First all the imports at the top, if you need new imports add here

In [11]:
import sys
import time
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import uuid
from urllib.parse import urljoin
import requests
import os
from datetime import datetime
import pandas as pd


Run the code below to open the browser

In [4]:
#set driver only once so it doesn't open a huge amount of browsers
driver = webdriver.Chrome()

Below is the new method using links instead of navigating through the filters with selenium the filters are applied through an f string url.

In [ ]:
#the same function that we made earlier to extract data, but without the next page button click, this is now handled through the url
def extract_ranking_page(driver, city, gender, hyrox_type, collection_id):
    #read the page that is open right now and return its rows as a list
    soup = BeautifulSoup(driver.page_source, "html.parser")
    rows = soup.select('div[data-sex] ul.list-group > li.list-group-item:not(.list-group-header)')

    page_results = []

    for row in rows:
        flag = row.find("img", class_="nation__icon")
        abbreviation = flag.get("alt") if flag else None

        link_tag = row.find("a", href=True)
        personal_link = urljoin(driver.current_url, link_tag["href"]) if link_tag else None #we need the full link instead of the relative link otherwise it did not woerk

        age_field = row.find("div", class_="type-age_class")
        age_group = age_field.get_text(" ", strip=True).replace("Age Group", "").strip() if age_field else None

        time_field = row.find("div", class_="type-time")
        total_time = time_field.get_text(" ", strip=True).replace("Total", "").strip() if time_field else None

        page_results.append({
            "city_of_event": city,
            "hyrox_type": hyrox_type,
            "gender_athlete": gender,
            "nationality_athlete": abbreviation,
            "age_group": age_group,
            "total_time": total_time,
            "personal_link": personal_link,
            "collection_id": collection_id,
            "record_id": str(uuid.uuid4()),
        })

    return page_results

#If we want to check different cities etc. simply change them here
cities_to_check = ["AMS26", "GRQ26", "BRU26", "ROT26", "MAASTRICHT25", "Utrecht2025", "GNE25"]
genders_to_check = ["M", "W"]
hyrox_type = "OVERALL"

#one list for everything, so it goes outside all the loops
results_ranking_page = []

#one id for this whole run
collection_id = str(uuid.uuid4())

import os
from datetime import datetime

#make the data/raw folder in the main project folder if for some reason it deosn't exist anymore.
os.makedirs("../data/raw", exist_ok=True)

#some date stuff to give every run its own file so a new run never overwrites an older one
run_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
csv_name = f"../data/raw/hyrox_results_{run_time}.csv"
json_name = f"../data/raw/hyrox_results_{run_time}.json"

#one list for everything, so it goes outside all the loops
results_ranking_page = []

#one id for this whole run
collection_id = str(uuid.uuid4())

#The for loops loop over cities and genders to check
for city in cities_to_check:
    for gender in genders_to_check:
        print("Now collecting:", city, gender)

        #start at page 1 again for every city and gender
        page = 1

        #The while loop which keeps going until a page has less <100 results or if the page is completely empty
        while True:
            url = f"https://results.hyrox.com/season-8/?page={page}&event=H_{city}_{hyrox_type}&num_results=100&pid=list&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D={gender}&search%5Bage_class%5D=%25&search%5Bnation%5D=%25"
            driver.get(url)
            time.sleep(2)

            #let the function defined earlier read the page
            page_results = extract_ranking_page(driver, city, gender, hyrox_type, collection_id)
            print("Page", page, "- result rows found:", len(page_results))

            #add this page's rows to the big list
            results_ranking_page.extend(page_results)

            #save everything we have so far every time so we don't lose anything if somethign goes wrong
            #this overwrites the file each time so it is always up to date
            pd.DataFrame(results_ranking_page).to_csv(csv_name, index=False, encoding="utf-8")
            pd.DataFrame(results_ranking_page).to_json(json_name, orient="records", indent=2, force_ascii=False)

            #stop if the page was empty or not full, that means it was the last page
            if len(page_results) < 100:
                break

            page += 1

print("Total rows collected:", len(results_ranking_page))
print("Saved to:", csv_name, "and", json_name)

Now collecting: AMS26 M
Page 1 - result rows found: 100
Page 2 - result rows found: 100
Page 3 - result rows found: 100
Page 4 - result rows found: 100
Page 5 - result rows found: 100
Page 6 - result rows found: 100
Page 7 - result rows found: 100
Page 8 - result rows found: 100
Page 9 - result rows found: 100
Page 10 - result rows found: 100
Page 11 - result rows found: 100
Page 12 - result rows found: 100
Page 13 - result rows found: 100
Page 14 - result rows found: 100
Page 15 - result rows found: 100
Page 16 - result rows found: 100
Page 17 - result rows found: 100
Page 18 - result rows found: 100
Page 19 - result rows found: 100
Page 20 - result rows found: 100
Page 21 - result rows found: 100
Page 22 - result rows found: 100
Page 23 - result rows found: 100
Page 24 - result rows found: 100
Page 25 - result rows found: 100
Page 26 - result rows found: 100
Page 27 - result rows found: 100
Page 28 - result rows found: 98
Now collecting: AMS26 W
Page 1 - result rows found: 100
Page 2